### Step1 - Project Structure

```text
btc-ml-pipeline/
│
├── collector/
│   └── stream.py
│
├── processor/
│   ├── aggregator.py
│   └── features.py
│
├── storage/
│   ├── raw/
│   ├── candles/
│   └── features/
│
├── config/
│   └── settings.py
│
├── docker/
│   └── docker-compose.yml
│
├── app/
│   └── dashboard.py
│
└── main.py
```

### step2 Data Model (Define First)

```json
{
    "timestamp": int,
    "price": float,
    "quantity": float,
    "is_buyer_maker": bool,
    "trade_id": int
}
```

### Step 3 — Collector (WebSocket → Stream Buffer)

___

- Create `collector/stream.py`
   - connect to Binance
   - convert raw message → structured record
   - push into in-memory queue (later Kafka)

```python
import asyncio
import json
import websockets
from datetime import datetime
from collections import deque

WS_URL = "wss://stream.binance.com:9443/ws/btcusdt@trade"

buffer = deque(maxlen=10000)

async def collect():
    async with websockets.connect(WS_URL) as ws:
        print("Connected")

        while True:
            msg = await ws.recv()
            data = json.loads(msg)

            record = {
                "timestamp": data["T"],
                "price": float(data["p"]),
                "quantity": float(data["q"]),
                "is_buyer_maker": data["m"],
                "trade_id": data["t"]
            }

            buffer.append(record)

            print(record)

asyncio.run(collect())
```

___

### Step 4 — Build 1-Second Candle Aggregator

- Create: `processor/aggregator.py`

| field  | meaning      |
| ------ | ------------ |
| open   | first trade  |
| high   | max price    |
| low    | min price    |
| close  | last trade   |
| volume | sum quantity |


**Aggregation logic**

```python
from collections import defaultdict
import time

class CandleBuilder:
    def __init__(self):
        self.current_bucket = None
        self.data = defaultdict(list)

    def add_trade(self, trade):
        bucket = trade["timestamp"] // 1000  # 1-second bucket

        if self.current_bucket is None:
            self.current_bucket = bucket

        if bucket != self.current_bucket:
            candle = self.build_candle(self.current_bucket)
            self.data[self.current_bucket] = []
            self.current_bucket = bucket
            return candle

        self.data[bucket].append(trade)
        return None

    def build_candle(self, bucket):
        trades = self.data[bucket]

        if not trades:
            return None

        prices = [t["price"] for t in trades]
        volumes = [t["quantity"] for t in trades]

        return {
            "timestamp": bucket,
            "open": prices[0],
            "high": max(prices),
            "low": min(prices),
            "close": prices[-1],
            "volume": sum(volumes)
        }
```

___

### Step 5 — Storage Layer (VERY IMPORTANT)

- Raw trades → Parquet
- Candles → Parquet

**install:**
```bash
pip install pandas pyarrow
```

#### Save function

```python
import pandas as pd
import os

def save_parquet(data, path):
    df = pd.DataFrame(data)

    os.makedirs(os.path.dirname(path), exist_ok=True)

    df.to_parquet(path, index=False)
```

___

### Step 6 — First Feature (Simple but Powerful)

- return

 $$return_t = \frac{close_t - close_{t-1}}{close_{t-1}}$$

**Implementation**

```python
def add_return(df):
    df["return"] = df["close"].pct_change()
    return df
```    

___

### Step 7 — Run Pipeline (Local Test)